In [0]:
from pyspark.sql.functions import trim, col, coalesce, lit, try_to_date
from pyspark.sql.types import DoubleType

In [0]:
df_stores = spark.table("retailer.bronze.stores_raw")

display(df_stores)

In [0]:
df_stores_clean = df_stores.drop("_file", "_line", "_modified")

In [0]:
df_stores_clean = (
    df_stores_clean
    .withColumn("country", trim(col("country")))
    .withColumn("state", trim(col("state")))
)

In [0]:
df_stores_clean = df_stores_clean.withColumn(
    "square_meters",
    coalesce(col("square_meters").cast(DoubleType()), lit(0.0))
)

In [0]:
df_stores_clean = df_stores_clean.withColumn(
    "open_date",
    coalesce(
        try_to_date(col("open_date"), "dd-MM-yyyy"),
        try_to_date(col("open_date"), "M/d/yyyy")
    )
)

In [0]:
df_stores_clean = (
    df_stores_clean
    .filter(col("store_key").isNotNull())
    .dropDuplicates(["store_key"])
)

In [0]:
display(df_stores_clean)

df_stores_clean.printSchema()

In [0]:
df_stores_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailer.silver.stores")